# Additional Federal water data

*Federal Open Water Data for Researchers* (CUAHSI), session 6.

**Learning goal:** Take a first step with related Federal water data products (SWOT discharge from the SWORD of Science and NOAA AORC precipitation) using the same discover-before-download habits as Module 3.

**Environment:** run this notebook in the `m04-synthesis` conda environment, created from `module-04_lesson-03_additional-federal-water-data_environment.yml` in this folder (see the bundle README). Run the cells in order: later cells reuse variables from earlier ones.

This lesson has no "Now you try it" task, so the student and solutions copies of this notebook are the same.

Now that you have worked with NASA Surface Water and Ocean Topography (SWOT) water surface elevation, NOAA National Water Model (NWM) streamflow, and USGS Water Data for the Nation (WDFN) discharge data products, you are more equipped to broaden your use of other Federal water data products and tools. Consider the following: _Note that other Federal agencies also produce or use water data, including the U.S. Department of Agriculture (USDA), the U.S. Army Corps of Engineers (USACE) and the Environmental Protection Agency (EPA)._

## Attribution and licence

This notebook is part of *Federal Open Water Data for Researchers*, a CUAHSI course, and is released under the [GNU General Public License v3.0](https://www.gnu.org/licenses/gpl-3.0.html) (GPL-3.0). You may use, change and share it under the same licence.

It adapts code and ideas from these sources:

- Adapted from [Collecting and Manipulating AORC Data](https://github.com/CUAHSI/notebooks/tree/develop/Data%20Access%20Examples/AORC%20-%20Data%20Collection%20and%20Manipulation%20Primer) by Tony Castronova, Irene Garousi-Nejad, Danielle Tijerina-Kreuzer and Abner Bogan, CUAHSI notebooks (GPL-3.0).
- Adapted from [SWOT - Visualizing SOS Discharge with Xarray](https://github.com/CUAHSI/notebooks/tree/develop/Data%20Access%20Examples/SWOT%20-%20Visualizing%20SOS%20Discharge%20with%20Xarray), CUAHSI notebooks (GPL-3.0).
- Adapted from [SWOT - Compare Observed and SoS Discharge](https://github.com/CUAHSI/notebooks/tree/develop/Data%20Access%20Examples/SWOT%20-%20Compare%20Observed%20and%20SoS%20Discharge), CUAHSI notebooks (GPL-3.0).

Other sources used in the lesson are listed under "Further reading" at the end of this notebook.

## Credentials

Credentials come only from your account or environment, never from the notebook itself.

**NASA Earthdata Login.** Downloading SWOT files needs a free account from [urs.earthdata.nasa.gov](https://urs.earthdata.nasa.gov). `earthaccess.login()` looks for your username and password in the `EARTHDATA_USERNAME` and `EARTHDATA_PASSWORD` environment variables, then in a `~/.netrc` file, and otherwise asks you for them. In a hosted Jupyter environment, the simplest route is to let it ask: run the cell and type your details into the boxes it shows. To avoid typing them every session, run `earthaccess.login(persist=True)` once, which saves them to `~/.netrc` in your home folder. Never type your password into a code cell: it would be saved in the notebook. See the [`earthaccess` authentication how-to](https://earthaccess.readthedocs.io/en/latest/user/howto/authenticate/).

In [ ]:
import earthaccess

# Uses EARTHDATA_USERNAME/EARTHDATA_PASSWORD or ~/.netrc if set; otherwise asks you to type them
auth = earthaccess.login()
print("Logged in to NASA Earthdata:", auth.authenticated)

## Exploring SWOT data in the browser: SWOTViz

Before writing code, it often helps to *look*: which SWOT reaches cover your river, how often they were observed, and whether the time series look sensible. [SWOTViz](https://swotviz.cuahsi.io/) is a CUAHSI web viewer for SWOT river and lake data. It shows SWORD reaches and nodes and Prior Lake Database lakes on a map, along with SWOT orbit tracks and pass counts, and plots a selected feature's time series from hydrocron, with options to download the data. It is a quick way to find reach IDs and check data coverage before you use the code patterns from Module 3. Its source code is on GitHub ([CUAHSI/SWOT-Data-Viewer](https://github.com/CUAHSI/SWOT-Data-Viewer), GPL-3.0).

## NASA SWOT discharge: the SWORD of Science (SoS)

> **In development.** SoS discharge estimates are still being developed and validated. The collection shown here is Version 3 (`SWOT_L4_HR_DAWG_SOS_DISCHARGE_V3`), covering May 2023 to May 2025. This course does not yet recommend it for research use: treat it as a preview of where SWOT discharge is heading, not as a data source for your analysis. It is fine to explore the files and learn their structure; don't use SoS values as results or as inputs in research until NASA marks them validated.

SWOT measures water surface elevation, width and slope, not discharge. Discharge is estimated afterwards by the SWOT Discharge Algorithm Working Group (DAWG), which runs several discharge algorithms over SWORD reaches in a cloud workflow called Confluence. The **SWORD of Science (SoS)** collects the results: the prior information each algorithm used (for example, gage records and model estimates) and the discharge estimates themselves, for every reach, organized by continent ([PO.DAAC SoS tutorial](https://podaac.github.io/tutorials/notebooks/datasets/SWOT_L4_DAWG_SOS_DISCHARGE.html)). It is a Level 4 product: model or algorithm output built from lower-level data.

In [ ]:
import earthaccess

earthaccess.login()
sos = earthaccess.search_data(short_name="SWOT_L4_HR_DAWG_SOS_DISCHARGE_V3")
for g in sos:
    print(g["umm"]["GranuleUR"], round(g.size / 1024, 1), "GB")  # g.size is in MB

Three things to notice before you download:

- **One file per continent, and they are large.** The North America file alone is about 9 GB, and Asia's about 27 GB. If you work outside the cloud, read only the groups and reaches you need (see the notebooks below), or work on a cloud JupyterHub in AWS `us-west-2`.
- **Know what is inside.** Each file is NetCDF with one group per algorithm (plus the priors and validation data), indexed by SWORD `reach_id`, with discharge in m³/s. `unconstrained` in the name means the algorithms ran without using gage data; "gauge-constrained" results (SWOT documents spell it "gauge") use historical gage records to calibrate the algorithms.
- **Check the time period and version.** These Version 3 files cover May 2023 to May 2025 (it's in the file name), so they do not include the December 2025 Skagit flood used in this module. `hydrocron` can also return SoS discharge fields (for example `sos_consensus_q`) for a single reach, using `collection_name=SWOT_L2_HR_RiverSP_2.0` ([Hydrocron timeseries docs](https://podaac.github.io/hydrocron/timeseries)). That is the Version C collection, a different version from the Version D data in the case study, so don't mix values from the two without checking. Cite the collection by its DOI ([10.5067/SWOT-SOS-RD3](https://doi.org/10.5067/SWOT-SOS-RD3)), and record the collection short name (it carries the version) and the file name with any SoS results, since each new version reprocesses the whole record.

**Go further:** the CUAHSI notebooks [SWOT - Visualizing SOS Discharge with Xarray](https://github.com/CUAHSI/notebooks/tree/develop/Data%20Access%20Examples/SWOT%20-%20Visualizing%20SOS%20Discharge%20with%20Xarray) and [SWOT - Compare Observed and SoS Discharge](https://github.com/CUAHSI/notebooks/tree/develop/Data%20Access%20Examples/SWOT%20-%20Compare%20Observed%20and%20SoS%20Discharge) open the SoS groups with `xarray` and compare SoS discharge with community gage data.

## NOAA AORC precipitation

The **Analysis of Record for Calibration (AORC)** is a gridded, hourly record of precipitation and other near-surface weather (temperature, humidity, wind, radiation) over the contiguous United States. The copy used here, in the NWM retrospective archive, is on the 1 km NWM grid. It is the historical weather forcing used to calibrate the National Water Model and to drive its retrospective simulation ([AWS Registry of Open Data: NWM archive](https://registry.opendata.aws/nwm-archive/)). That makes it the natural precipitation dataset to pair with NWM streamflow, and a good first look at what "caused" a hydrograph.

In [ ]:
import fsspec
import pyproj
import xarray as xr

store = fsspec.get_mapper("s3://noaa-nwm-retrospective-3-0-pds/CONUS/zarr/forcing/precip.zarr", anon=True)
aorc = xr.open_zarr(store)
print(aorc.sizes, aorc.time.values[[0, -1]])

# The grid is Lambert Conformal Conic; convert the gage location into grid coordinates
lcc = pyproj.CRS.from_proj4(aorc["RAINRATE"].attrs["proj4"])
# USGS 12200500 (Skagit River near Mount Vernon), from its USGS monitoring-location record
gage_lon, gage_lat = -122.3354, 48.4448
x, y = pyproj.Transformer.from_crs("EPSG:4326", lcc, always_xy=True).transform(gage_lon, gage_lat)

rain = aorc["RAINRATE"].sel(x=x, y=y, method="nearest").sel(time=slice("2022-12-01", "2022-12-31"))
mm_per_hour = (rain * 3600).load()  # RAINRATE is in mm/s
print(mm_per_hour.size, "hours, total", round(float(mm_per_hour.sum()), 1), "mm")

`aorc` is a lazy `xarray` Dataset of 1 km grid cells (`x`, `y` in the model's Lambert Conformal Conic projection, in meters) and hourly `time` steps. `RAINRATE` is the precipitation rate in mm/s, so multiplying by 3,600 gives millimeters per hour; `mm_per_hour` is a 744-value DataArray, one value per hour of December 2022 in UTC. AORC has no per-value quality flag. Its version and provenance are in the path: this is the copy published with the NWM v3.0 retrospective (`noaa-nwm-retrospective-3-0-pds`), so record that path with any results.

To see what the data look like, plot the hourly series:

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(mm_per_hour["time"].values, mm_per_hour.values, width=1 / 24, color="#2a78d6")  # one bar per hour
ax.set_ylabel("Precipitation (mm/h)")
ax.set_xlabel("Time (UTC)")
ax.set_title("AORC hourly precipitation at the grid cell nearest USGS 12200500, December 2022")
fig.autofmt_xdate()
plt.show()

The full dataset is about 27 TB, but this request reads only the few chunks that contain the one grid cell and one month. It still takes about a minute from a laptop, because each chunk holds 28 days for a 350 × 350 km block. This **retrospective** copy ends in January 2023, so it does not cover the December 2025 flood. For recent events, NOAA distributes AORC through other channels.

## Other agencies' water data: USDA and USACE

NASA, NOAA and USGS are not the only Federal sources of water data. The U.S. Department of Agriculture (USDA) and the U.S. Army Corps of Engineers (USACE) also publish water data that can complement the products in this course.

## Further reading

- [PO.DAAC SWOT Cookbook](https://podaac.github.io/tutorials/quarto_text/SWOT.html), including the [SoS discharge tutorial](https://podaac.github.io/tutorials/notebooks/datasets/SWOT_L4_DAWG_SOS_DISCHARGE.html).
- [NWM retrospective archive, including AORC forcing (AWS Registry of Open Data)](https://registry.opendata.aws/nwm-archive/).
- Adapted from [Collecting and Manipulating AORC Data](https://github.com/CUAHSI/notebooks/tree/develop/Data%20Access%20Examples/AORC%20-%20Data%20Collection%20and%20Manipulation%20Primer) by Tony Castronova, Irene Garousi-Nejad, Danielle Tijerina-Kreuzer and Abner Bogan, CUAHSI notebooks (GPL-3.0).
- Adapted from [SWOT - Visualizing SOS Discharge with Xarray](https://github.com/CUAHSI/notebooks/tree/develop/Data%20Access%20Examples/SWOT%20-%20Visualizing%20SOS%20Discharge%20with%20Xarray), CUAHSI notebooks (GPL-3.0).
- Adapted from [SWOT - Compare Observed and SoS Discharge](https://github.com/CUAHSI/notebooks/tree/develop/Data%20Access%20Examples/SWOT%20-%20Compare%20Observed%20and%20SoS%20Discharge), CUAHSI notebooks (GPL-3.0).

- [SWOTViz](https://swotviz.cuahsi.io/) (CUAHSI), source code at [CUAHSI/SWOT-Data-Viewer](https://github.com/CUAHSI/SWOT-Data-Viewer) (GPL-3.0).